In [ ]:
# hide
import numpy as np
import pyquist as pq
from scipy.signal import lfilter

In [ ]:
# collapse
def pluck(pitch: float, duration: float, f_s: int = 44100, **kwargs) -> pq.Audio:
    """A Karplus-Strong plucked string (Chapter 12) that decays by 60 dB in `duration`
    seconds. An all-pass filter (Chapter 13) in the loop tunes it precisely."""
    P = f_s / pq.helper.pitch_to_frequency(pitch)   # the period, in (fractional) samples
    M = int(P - 0.6)                                 # the delay line (the low-pass adds 1/2 sample)
    D = P - 0.5 - M                                  # the all-pass adds the remaining D samples
    c = (1 - D) / (1 + D)                            # the all-pass gain for a delay of D samples
    g = 10 ** (-3 * P / (f_s * duration))            # decay by 60 dB over the duration
    x = np.zeros(int(duration * f_s))
    x[:M] = np.random.uniform(-1.0, 1.0, M)          # excitation: a burst of white noise
    # The whole loop is one recursive filter (Chapter 9), which SciPy runs quickly:
    # y[n] = x[n] + c x[n-1] - c y[n-1] + (g/2) (c y[n-M] + (1 + c) y[n-M-1] + y[n-M-2])
    a = np.zeros(M + 3)
    a[:2] = [1.0, c]
    a[M:] -= 0.5 * g * np.array([c, 1 + c, 1.0])
    return pq.Audio(lfilter([1.0, c], a, x), f_s)

In [ ]:
def uniform(N: int, pitches: list[int] = [60, 62, 64, 67, 69], ioi: float = 0.25,
            duration: float = 0.5) -> pq.Score:
    """A melody of N notes, whose pitches are chosen uniformly at random.
    Onsets are in beats, and durations (decay times) are in seconds."""
    choices = np.random.choice(pitches, size=N)
    return pq.Score([(i * ioi, {"pitch": pitch, "duration": duration})
                     for i, pitch in enumerate(choices)])


score = uniform(32)
pq.play(score.render(pluck, metronome=pq.BasicMetronome(bpm=120)), normalize=True)